# Dump VideoMAE action labels for GIFs
This notebook runs a Hugging Face VideoMAE action model over GIFs and saves top‑k labels.
Update `GIF_DIR` to your folder of test GIFs, then run all cells.

In [ ]:
!pip -q install transformers pillow torch --upgrade

In [3]:
import os, glob, csv, json
import numpy as np
from PIL import Image, ImageSequence
import torch
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

MODEL_ID = "MCG-NJU/videomae-base-finetuned-kinetics"  # change if you use a different checkpoint
GIF_DIR = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images"  # <-- change this
PATTERN = "*.gif"
NUM_FRAMES = 16
TOPK = 5
OUT_CSV = "actions.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = VideoMAEImageProcessor.from_pretrained(MODEL_ID)
model = VideoMAEForVideoClassification.from_pretrained(MODEL_ID).to(device).eval()

import numpy as np
from PIL import Image, ImageSequence
import torch

TARGET_SIZE = (224, 224)

def load_gif_frames(gif_path, max_frames=None):
    im = Image.open(gif_path)
    frames = []
    for i, fr in enumerate(ImageSequence.Iterator(im)):
        if max_frames is not None and i >= max_frames:
            break
        fr = fr.convert("RGB").resize(TARGET_SIZE)   # <-- force 224x224
        frames.append(fr)
    return frames

def sample_or_pad(frames, num_frames=16):
    """
    Always return exactly num_frames frames:
    - If too many: evenly sample
    - If too few: repeat last frame
    """
    if len(frames) == 0:
        return []

    if len(frames) >= num_frames:
        idxs = np.linspace(0, len(frames)-1, num_frames).round().astype(int).tolist()
        return [frames[i] for i in idxs]
    else:
        out = frames[:]
        while len(out) < num_frames:
            out.append(out[-1])
        return out

@torch.no_grad()
def predict_topk(frames, topk=5):
    frames = sample_or_pad(frames, num_frames=16)

    inputs = processor(frames, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)
    probs = torch.softmax(logits, dim=-1)

    vals, idxs = torch.topk(probs, k=min(topk, probs.numel()))
    out = []
    for r, (i, v) in enumerate(zip(idxs.tolist(), vals.tolist()), start=1):
        out.append((r, model.config.id2label.get(i, str(i)), float(v)))
    return out


gif_paths = sorted(glob.glob(os.path.join(GIF_DIR, PATTERN)))
print("Found", len(gif_paths), "GIFs")


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

Found 6142 GIFs


In [4]:
rows = []
for p in gif_paths:
    frames = load_gif_frames(p)
    preds = predict_topk(frames, topk=TOPK)
    for (r,lab,sc) in preds:
        rows.append([p, r, lab, sc])
    print(os.path.basename(p), "=>", preds[:3], "..." if len(preds)>3 else "")


1000WjcUQeqOaY.gif => [(1, 'sign language interpreting', 0.1443415731191635), (2, 'stretching arm', 0.03220106288790703), (3, 'smoking', 0.022495731711387634)] ...
100KE3jcx3ZKM.gif => [(1, 'singing', 0.14490972459316254), (2, 'crying', 0.05235402658581734), (3, 'answering questions', 0.05225730314850807)] ...
100YmlniUkSv84.gif => [(1, 'kissing', 0.07222172617912292), (2, 'headbutting', 0.029856961220502853), (3, 'snorkeling', 0.02505236119031906)] ...
100v69Oj1OFjC8.gif => [(1, 'crying', 0.4536580741405487), (2, 'playing chess', 0.06327597051858902), (3, 'answering questions', 0.046234454959630966)] ...
101htdn61ek6QM.gif => [(1, 'laughing', 0.5322050452232361), (2, 'answering questions', 0.02056051231920719), (3, 'eating hotdog', 0.013759909197688103)] ...
102JEsI5tKRsfm.gif => [(1, 'tossing coin', 0.11533922702074051), (2, 'playing chess', 0.07265787571668625), (3, 'crying', 0.07175754755735397)] ...
102QyB4Sar1AeA.gif => [(1, 'kissing', 0.17150577902793884), (2, 'shaving head', 0.

OSError: image file is truncated (155 bytes not processed)

In [ ]:
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file","rank","label","score"])
    w.writerows(rows)
print("Saved:", OUT_CSV)
